<a href="https://colab.research.google.com/github/sngillard/student-outcome-classification/blob/main/notebooks/04_gradient_boosting_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Load processed dataset from GitHub URL
df = pd.read_csv('https://raw.githubusercontent.com/sngillard/student-outcome-classification/refs/heads/main/data/processed/student_data_processed_2026-01-08.csv')

# Split dataset into features and target
X = df.drop(columns=['Target'])
y = df['Target']

# Split data into training and testing sets with 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)

Training shape: (3539, 34)
Testing shape: (885, 34)


In [ ]:
# Create a Gradient Boosting Classifier model with stochastic subsampling for optimization
from sklearn.ensemble import GradientBoostingClassifier

# Build Gradient Boosting Classifier model with stochastic subsampling
gb_classifier_model = GradientBoostingClassifier(random_state=42, subsample=0.8)

In [ ]:
# Train the model using the AI/ML algorithm.
gb_classifier_model.fit(X_train, y_train)

# Test the model runs/makes classification predictions
y_prediction_gb = gb_classifier_model.predict(X_test)
print('Testing predictions:', y_prediction_gb[:20])

Testing predictions: [2 2 0 2 2 2 2 2 2 2 2 1 0 0 1 1 1 0 0 0]


The above predictions make sense based on the dataset distribution for the target variable that can be viewed in the bar chart found in notebook 02_exploratory_data_analysis.ipynb

In [ ]:
# Evaluate model accuracy using these metrics: accuracy, precision, recall, f1-score, and confusion matrix.
y_prediction_gb = gb_classifier_model.predict(X_test)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

accuracy_score_gb = accuracy_score(y_test, y_prediction_gb)
precision_score_gb= precision_score(y_test, y_prediction_gb, average='weighted')
recall_score_gb = recall_score(y_test, y_prediction_gb, average='weighted')
f1_score_gb= f1_score(y_test, y_prediction_gb, average='weighted')
confusion_matrix_gb = confusion_matrix(y_test, y_prediction_gb)

print('Gradient Boosting (with Stochastic Subsampling) Model- Evaluation Metrics:')
print(f'Accuracy: {accuracy_score_gb:.4f}')
print(f'Precision (weighted): {precision_score_gb:.4f}')
print(f'Recall (weighted): {recall_score_gb:.4f}')
print(f'F1 Score (weighted): {f1_score_gb:.4f}')
print('Confusion Matrix for Logistic Gradient Boosting Model:')
print(confusion_matrix_gb)
print(classification_report(y_test, y_prediction_gb))

Gradient Boosting (with Stochastic Subsampling) Model- Evaluation Metrics:
Accuracy: 0.7605
Precision (weighted): 0.7473
Recall (weighted): 0.7605
F1 Score (weighted): 0.7495
Confusion Matrix for Logistic Gradient Boosting Model:
[[209  32  43]
 [ 38  60  61]
 [ 13  25 404]]
              precision    recall  f1-score   support

           0       0.80      0.74      0.77       284
           1       0.51      0.38      0.43       159
           2       0.80      0.91      0.85       442

    accuracy                           0.76       885
   macro avg       0.70      0.68      0.68       885
weighted avg       0.75      0.76      0.75       885



### Results from metrics for Gradient Boosting Classifier Model (with Stochastic subsampling):
* **Accuracy**: 0.7605
* **Precision**: 0.7473
* **Recall**:0.7605
* **F1-Score:** 0.7495
* **Confusion Matrix**: <br>
 209        32          43 <br>
 38        60          61 <br>
13          25          404
*  **Classification matrix**: *see above*


### Model Evaluation summary:
* Before cross-validation and hyperparameter tuning, the models have similar performance based on the metrics above.
* The gradient boosting model has *slightly* better accuracy than the multinomial logistic regression model.

In [ ]:
# Apply cross-validation techniques.
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
from sklearn.ensemble import  GradientBoostingClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform cross-validation with accuracy score
cv_scores_gb = cross_val_score(gb_classifier_model, X_train, y_train, cv=skf, scoring='accuracy')

print('Gradient boosting cross-validation scores:', cv_scores_gb)
print('Gradient boosting mean cross-validation accuracy:', np.mean(cv_scores_gb))
print('Gradient boosting cross-validation standard deviation:', np.std(cv_scores_gb))

Gradient boosting cross-validation scores: [0.77824859 0.78389831 0.78248588 0.77118644 0.76803395]
Gradient boosting mean cross-validation accuracy: 0.7767706310582632
Gradient boosting cross-validation standard deviation: 0.006215544344264626


### Comparing Models After Cross-Validation
* Gradient boosting **cross-validation scores**: [0.77542373 0.78389831 0.78248588 0.76836158 0.7708628 ]
* Logistic regression **cross-validation scores**: [0.7740113  0.7740113  0.76949153 0.75706215 0.74660633]
* Gradient boosting **mean cross-validation accuracy**: 0.7762064584182389
* Logistic regression **mean cross-validation accuracy**: 0.7642365212056139
* Gradient boosting **cross-validation standard deviation**: 0.006153129589961597
* Logistic regression **cross-validation standard deviation**: 0.010779636044061494

Applying stratified 5-fold cross-validation helped to analyze the performance of each model.
   
The **gradient boosting model** produced a slightly **higher mean cross-validation accuracy** with **lower variance** shown in the std score compared to the multinomial logistic regression model.

These findings indicate that the **gradient boosting model has a *slightly* more consistent performance across folds**.

**Note** Hyperparameter tuning may take 2-3 minutes to run.

In [ ]:
# Use hyperparameter tuning to optimize the model.

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV # Using RandomizedSearchCV instead of GridSearchCV, as GridSearchCV had VERY slow execution
from scipy.stats import randint, uniform

# Build Gradient Boosting Classifier model
gb_classifier_model = GradientBoostingClassifier(random_state=42)

param_distributions_gb = {
    'n_estimators': randint(80, 200),
    'learning_rate': uniform(0.03, 0.15),
    'max_depth': randint(2, 5),
    'subsample': uniform(0.7, 0.25),
}

cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search_gb = RandomizedSearchCV(
    estimator=gb_classifier_model,
    param_distributions=param_distributions_gb,
    n_iter=10,
    scoring='accuracy',
    cv = cross_val,
    random_state=42,
    n_jobs = -1,
    verbose=1
)

random_search_gb.fit(X_train, y_train)

print('Best gradient boosting params:', random_search_gb.best_params_)
print('Best gradient boosting cross-validation accuracy:', random_search_gb.best_score_)

best_gb_model = random_search_gb.best_estimator_

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best gradient boosting params: {'learning_rate': np.float64(0.1707829063523625), 'max_depth': 3, 'n_estimators': 143, 'subsample': np.float64(0.9480528898228043)}
Best gradient boosting cross-validation accuracy: 0.779031317175301


# Hyperparameter Tuning- Gradient Boosting Model
- RandomizedSearchCV was used to tune the hyperparameters to improve runtime.

### RandomizedSearchCV was used to optimize the gradient boosting classifier model.

### Best Tuned hyperparameters- Gradient boosting model (with sample value from running the algorithm):
- Number of estimators: n_estimators = 143
- Learning rate: learning_rate = 0.171
- Maximum tree depth: max_depth = 3
- Subsample rate: subsample = 0.95

### Best cross-validation accuracy: 0.78